# Tutorial 4: GRN Analysis

After training, we can extract and analyze the learned Gene Regulatory Network. The model learns two regulatory matrices:

- **E1 (TF→Region)**: Which TFs bind to which regulatory regions
- **E2 (Region→Gene)**: Which regions regulate which genes

This tutorial shows how to extract these matrices and find key regulatory relationships.

## Setup

In [ ]:
import torch

import deepscenic as ds

print(f"deepSCENIC version: {ds.__version__}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Load Model and Data

In [ ]:
data_dir = "data/MM_lines/"

# Load trained model and data
model = ds.tl.load_model(data_dir + "model.pt", device=device)
mdata = ds.read(data_dir + "preprocessed_data.h5mu")

print(f"Model: {len(model.tf_names)} TFs, {len(model.gene_names)} genes")

## Biological Context: Melanoma Cell States

The MM_lines dataset contains melanoma cells in three phenotypic states:

- **MEL (Melanocytic)**: Differentiated, proliferative cells expressing melanocyte markers
- **MES (Mesenchymal)**: Invasive, therapy-resistant cells with EMT-like features
- **INT (Intermediate)**: Transitional cells between MEL and MES states

### Key TFs to Look For

Based on the SCENIC+ paper, we expect certain TFs to be active in specific cell states:

| Cell State | Expected Active TFs |
|------------|-------------------|
| **MEL** | SOX10, MITF, TFAP2A, ZEB2 |
| **MES** | JUN, NFIB, ZEB1, NR2F2 |
| **INT** | SOX6, EGR3, ETV4 |

As you explore the GRN below, look for these patterns in the TF activity heatmaps.

## Extract GRN

The GRN consists of two matrices:

- **E1 (TF->Region)**: TF binding potential to regulatory regions
- **E2 (Region->Gene)**: Region regulatory effect on genes

The combined GRN is E1 @ E2, giving TF->Gene regulatory weights.

In [ ]:
# Extract the full GRN as a DataFrame
# Returns dict with 'E1', 'E2', 'combined' DataFrames
grn = ds.tl.extract_grn(model)

print(f"E1 (TF→Region): {grn['E1'].shape}")
print(f"E2 (Region→Gene): {grn['E2'].shape}")
print(f"Combined (TF→Gene): {grn['combined'].shape}")

# Preview top regulatory edges
grn["combined"].head(10)

## Find TF Targets

Get the target genes for a specific TF, ranked by regulatory weight:

In [ ]:
# Find targets of a TF
# Uses Otsu thresholding to identify significant links
targets = ds.tl.get_tf_targets(model, tf_name="SOX10", top_k=20)
targets

### Biological Interpretation

SOX10 is a master regulator of the melanocytic lineage. Its target genes should include:
- **Melanocyte differentiation genes**: MITF, DCT, TYR, PMEL
- **Pigmentation pathway genes**: Genes involved in melanin synthesis
- **Other MEL-state TFs**: Creating feedforward regulatory loops

If SOX10's predicted targets don't include melanocyte markers, this may indicate issues with the GRN learning.

## Find Gene Regulators

Get the TFs that regulate a specific gene:

In [ ]:
# Find regulators of a gene
regulators = ds.tl.get_gene_regulators(model, gene_name="MITF", top_k=10)
regulators

## Visualize GRN Heatmaps

Heatmaps show the learned regulatory weights as matrices.

In [ ]:
# E1 heatmap (TF->Region)
ds.pl.heatmap_e1(model, top_k=20)

In [ ]:
# E2 heatmap (Region->Gene)
ds.pl.heatmap_e2(model, top_k=20)

In [ ]:
# Combined GRN heatmap (TF->Gene)
ds.pl.heatmap_grn(model, top_k=30)

## Cell-Type Aware GRN Analysis

The basic GRN extraction averages across all cells. For cell-type-specific regulatory networks, use the cell-type aware workflow:

1. Compute enhancer activity per cell type
2. Identify active enhancers using Otsu thresholding
3. Compute TF activity scores weighted by active enhancers
4. Identify key TFs per cell type
5. Build focused GRN for key TFs

In [ ]:
# Step 1: Compute mean enhancer activity per cell type
enh_activity = ds.tl.compute_celltype_enhancer_activity(
    model, mdata, celltype_key="cell_state"
)
print(f"Enhancer activity: {enh_activity.shape}")  # (n_regions, n_celltypes)

In [ ]:
# Step 2: Identify active enhancers per cell type (Otsu threshold)
active_enhancers = ds.tl.identify_active_enhancers(enh_activity)

for ct, regions in active_enhancers.items():
    print(f"  {ct}: {len(regions)} active enhancers")

In [ ]:
# Step 3: Visualize enhancer overlap across cell types
ds.pl.upset_active_enhancers(active_enhancers)

In [ ]:
# Step 4: Compute TF activity scores weighted by active enhancers
tf_scores = ds.tl.compute_tf_activity_scores(
    model, mdata, celltype_key="cell_state", active_enhancers=active_enhancers
)

# Visualize as heatmap
ds.pl.heatmap_celltype_activity(tf_scores, top_k=50)

### Interpreting the TF Activity Clustermap

In the clustermap above, look for:

1. **Cell-state-specific TF clusters**: TFs that are highly active in one state but not others
2. **Master regulators**: SOX10 and MITF should cluster together and show high activity in MEL cells
3. **MES regulators**: JUN-family TFs and ZEB1 should be active in MES cells
4. **Intermediate regulators**: If visible, INT cells may show a mix or have their own specific TFs

The hierarchical clustering on the left groups TFs by similar activity patterns across cell states, revealing the regulatory programs that define each phenotype.

In [ ]:
# Step 5: Identify key TFs (union across cell types)
key_tfs = ds.tl.identify_key_tfs(tf_scores)
print(f"Identified {len(key_tfs)} key TFs: {key_tfs[:10]}...")

In [ ]:
# Step 6: Build focused GRN for key TFs only
focused_grn = ds.tl.build_grn_for_tfs(model, key_tfs)
print(f"Focused GRN: {len(focused_grn)} edges")
focused_grn.head(10)

## Network Visualization

Network graphs show the GRN structure as connected nodes.

In [ ]:
# Visualize full GRN as network
ds.pl.network_grn(model, top_k=50)

In [ ]:
# TF-centric network: show targets of a specific TF
ds.pl.network_tf_targets(model, tf="SOX10", top_k=20)

In [ ]:
# Gene-centric network: show regulators of a specific gene
ds.pl.network_gene_regulators(model, gene="MITF", top_k=10)

## Next Steps

Now that you've extracted and visualized the GRN:

- Simulate perturbation experiments: {doc}`05_perturbation_analysis`
- Explore advanced genomic visualizations: {doc}`06_advanced_visualization`